# Importing

In [ ]:
import landbosse
import floris_cupy
import os
import cupy as cp
import numpy as np
import pandas as pd

from landbosse.main_function import run_landbosse # type: ignore

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import yaml
from scipy.stats import weibull_min
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union
from floris import FlorisModel
from floris.wind_data import WindRose
import floris as _floris_pkg

# Path to FLORIS built-in default GCH config
_FLORIS_DEFAULT_YAML = str(
	pathlib.Path(_floris_pkg.__file__).parent / 'default_inputs.yaml'
)
print('FLORIS', _floris_pkg.__version__, '-- default yaml found:', _FLORIS_DEFAULT_YAML)
import numpy as np
from scipy.optimize import differential_evolution

from scipy.optimize import minimize
import numpy as np

FLORIS 4.6.4 -- default yaml found: /home/lavender/Studies/Design of Wind Farms/.venv/lib/python3.12/site-packages/floris/default_inputs.yaml


In [ ]:
os.chdir(r"/home/lavender/Studies/Design of Wind Farms/Assignments/Assignment6/modular")

# Fixed Vars

In [ ]:
d_ws = 0.17
d_ti = 12/100
d_fuel_cost = 9.5
d_line_freq = 50
d_standard_V = 220
d_interconnect_V = 100
d_rent = 15000
d_o_and_m = 0.012
d_discount = 3.6/1e2
d_life_time = 20
d_construction_time = 12		# Month

TURB_COST_PER_MW = 1.3 * 1e6

In [ ]:
import numpy as np

rho = 1.225          # kg/m^3
D = 156.0            # m
V_rated = 9.0        # m/s (where power first reaches 3300 kW)
Ct_rated = 0.8027619595

A = np.pi * D**2 / 4
T_rated = 0.5 * rho * A * Ct_rated * V_rated**2

# Helper Funcs

In [ ]:
lattitude_degree_to_km = 111
longitude_degree_to_km_at_55 = 63

In [ ]:
def get_ref(stri):
	# This is using the DMS format - Degree Minutes Seconds
	
	lattLB = 55 + (14/60) + (39.2/(3600))
	longLB = 9 + (00/60) + (41.3/(3600))

	latt_deg = float(stri.strip().split("°")[0])
	latt_min = float(stri.strip().split("'")[0].split("°")[1])
	latt_sec = float(stri.strip().split("\"")[0].split("°")[1].split("'")[1].replace("\\", ""))

	long_deg = float(stri.strip().split("N")[1].split("°")[0])
	long_min = float(stri.strip().split("N")[1].split("'")[0].split("°")[1])
	long_sec = float(stri.strip().split("N")[1].split("\"")[0].split("°")[1].split("'")[1].replace("\\", ""))

	latt = latt_deg + (latt_min/60) + (latt_sec/(3600))
	long = long_deg + (long_min/60) + (long_sec/(3600))
 
	latt = (latt - lattLB) * lattitude_degree_to_km
 
	long = (long - longLB) * longitude_degree_to_km_at_55

	return latt, long

In [ ]:
def get_ref_WGS84(latt, long):
	# This is using the WGS84 lattitude/longitude form.
	
	lattLB = 55 + (14/60) + (39.2/(3600))
	longLB = 9 + (00/60) + (41.3/(3600))
 
	latt = (latt - lattLB) * lattitude_degree_to_km * 1000
 
	long = (long - longLB) * longitude_degree_to_km_at_55 * 1000

	return latt, long

# Functions

## Extracting Positions

In [ ]:
db = r"optimizedLayout/BAR_BAU_IEA_3.3MW_16_layout.txt"
db = r"/home/lavender/Studies/Design of Wind Farms/Assignments/Assignment6/modular/optimizedLayout/BAR_BAU_IEA_3.3MW_18_layout.txt"

dir = r"optimizedLayout"

In [ ]:
def readFile(db):
	def linetoxy(string):
		string=string.replace("x=", "").replace("y=", "").replace(" m", "")
		return [float(i) for i in string.replace(",", "").split(":")[1].strip().split()]

	substation_lines = []
	turbine_lines = []

	with open(db) as f:
		line = f.readline().strip()
		while line is not None:
			if "Substation" in line:
				line = f.readline().strip()
				substation_lines.append(linetoxy(line))
			if "Turbine" in line and "Positions" not in line:
				turbine_lines.append(linetoxy(line))
	
			line = f.readline().strip()
			
			if "Start" in line:
				break
	
	return turbine_lines, substation_lines